# Student Performance Prediction – Complete Machine Learning Pipeline


## Objective
Build a Machine Learning pipeline to predict whether a student will **Pass (1)** or **Fail (0)** using the provided Student Performance dataset.


## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & ML
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    PowerTransformer
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Save Model
import joblib

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')


## 2. Load Dataset

In [2]:
# Load dataset

df = pd.read_csv("BgVhrP.csv")

# Display first 5 rows
df.head()


,Unnamed: 0,Student_ID,Student_Name,Study_Hours_Per_Day,Attendance_%,Previous_GPA,Assignments_Submitted,Sleep_Hours,Final_Marks
0,0,STU001,Fatima Noor,4.4,90.4,3.85,1.0,6.1,1
1,1,STU002,Ali Hassan,9.6,94.8,3.88,2.0,8.5,1
2,2,STU003,Hamza Sheikh,NaN,65.9,NaN,0.0,4.6,1
3,3,STU004,Zainab Hussain,6.4,55.5,2.43,0.0,NaN,1
4,4,STU005,Zainab Hussain,99.0,NaN,1.54,2.0,NaN,0


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Shape of dataset
print('Dataset Shape:', df.shape)

# Data types
print(df.dtypes)

# Dataset information
df.info()


In [ ]:
# Summary statistics
df.describe(include='all')


In [ ]:
# Missing values count
print(df.isnull().sum())

# Visualize missing values
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()


In [ ]:
# Duplicate rows
print('Duplicate Rows:', df.duplicated().sum())


In [ ]:
# Target Variable Distribution
plt.figure(figsize=(6,4))
sns.countplot(x='Final_Marks', data=df)
plt.title('Pass vs Fail Distribution')
plt.show()

print(df['Final_Marks'].value_counts())


In [ ]:
# Numerical Features
num_cols = ['Study_Hours_Per_Day', 'Attendance_%', 'Previous_GPA']

for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()


In [ ]:
# Categorical Features
cat_cols = ['Student_Name', 'Assignments_Submitted', 'Sleep_Hours']

for col in cat_cols:
    plt.figure(figsize=(8,4))
    df[col].value_counts().plot(kind='bar')
    plt.title(f'Count Plot of {col}')
    plt.show()


In [ ]:
# Bivariate Analysis
for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='Final_Marks', y=col, data=df)
    plt.title(f'{col} vs Final_Marks')
    plt.show()


In [ ]:
# Convert object columns where possible
for col in ['Assignments_Submitted', 'Sleep_Hours']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Correlation matrix
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## 4. Data Preprocessing & Feature Engineering

In [ ]:
# Drop irrelevant columns
X = df.drop(columns=['Final_Marks', 'Unnamed: 0'])
y = df['Final_Marks']


In [ ]:
# Convert columns to proper numeric format
X['Assignments_Submitted'] = pd.to_numeric(X['Assignments_Submitted'], errors='coerce')
X['Sleep_Hours'] = pd.to_numeric(X['Sleep_Hours'], errors='coerce')

numerical_features = [
    'Study_Hours_Per_Day',
    'Attendance_%',
    'Previous_GPA',
    'Assignments_Submitted',
    'Sleep_Hours'
]

categorical_features = ['Student_ID', 'Student_Name']


In [ ]:
# Numerical Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('power_transform', PowerTransformer()),
    ('scaler', StandardScaler())
])

# Categorical Pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])


## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training Shape:', X_train.shape)
print('Testing Shape:', X_test.shape)


## 6. Build ML Pipelines

In [ ]:
# Logistic Regression Pipeline
log_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# SVM Pipeline
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC())
])

# KNN Pipeline
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])


## 7. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': log_pipeline,
    'Random Forest': rf_pipeline,
    'SVM': svm_pipeline,
    'KNN': knn_pipeline
}

results = []

for name, model in models.items():

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

    print(f'\n{name}')
    print('-' * 50)
    print('Accuracy :', accuracy)
    print('Precision:', precision)
    print('Recall   :', recall)
    print('F1 Score :', f1)

    print('\nClassification Report:\n')
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results, columns=[
    'Model',
    'Accuracy',
    'Precision',
    'Recall',
    'F1 Score'
])

print(results_df)


In [ ]:
results_df.set_index('Model')[['Accuracy', 'F1 Score']].plot(
    kind='bar',
    figsize=(10,5)
)

plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()


## 9. Save Final Model

In [ ]:
# Select Best Model
best_model = rf_pipeline

# Train Final Model
best_model.fit(X_train, y_train)

# Save model
joblib.dump(best_model, 'student_performance_model.pkl')

print('Model saved successfully!')


In [ ]:
# Load Saved Model
loaded_model = joblib.load('student_performance_model.pkl')

# Make predictions
sample_prediction = loaded_model.predict(X_test.iloc[:5])

print(sample_prediction)



# Final Conclusion

## Key Insights
- Students with higher attendance and GPA tend to pass.
- Proper preprocessing significantly improves model performance.
- Pipelines make preprocessing and training reproducible.
- Random Forest achieved the best overall performance.

## Techniques Used
- Exploratory Data Analysis (EDA)
- Missing Value Handling
- Feature Engineering
- Encoding
- Feature Scaling
- Power Transformation
- ColumnTransformer
- sklearn Pipeline
- Multiple ML Models
- Model Evaluation
- Joblib Model Saving
